# 8. Planner

*Using Microsoft Semantic Kernel (Agent Framework)*

Implements a planning agent that breaks down complex tasks into smaller steps, executes them, and tracks progress. This architecture includes integration with search tools for information retrieval.

In [ ]:
import os
from typing import Annotated
from dotenv import load_dotenv
import semantic_kernel as sk
from semantic_kernel.connectors.ai.open_ai import AzureChatCompletion
from semantic_kernel.contents import ChatHistory
from semantic_kernel.functions import kernel_function
from semantic_kernel.connectors.ai.open_ai.prompt_execution_settings.azure_chat_prompt_execution_settings import (
    AzureChatPromptExecutionSettings,
)
from semantic_kernel.connectors.ai.function_choice_behavior import FunctionChoiceBehavior

load_dotenv()

kernel = sk.Kernel()
service_id = "chat-gpt"
kernel.add_service(
    AzureChatCompletion(
        service_id=service_id,
        deployment_name="gpt-4o-mini",
        endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
        api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    )
)

In [ ]:
# Define planning and execution tools
class PlanningPlugin:
    @kernel_function(name="create_plan", description="Create a step-by-step plan for a task")
    def create_plan(self, task: Annotated[str, "Task to plan for"]) -> str:
        return f"""Plan for '{task}':
1. Research the topic
2. Gather necessary information
3. Organize findings
4. Create deliverable
5. Review and finalize"""

class SearchPlugin:
    @kernel_function(name="search", description="Search for information on a topic")
    def search(self, query: Annotated[str, "Search query"]) -> str:
        # Simulated search - in production, use actual search API like Tavily
        return f"Search results for '{query}': Found relevant information about {query} including key concepts and best practices."

class ExecutionPlugin:
    @kernel_function(name="execute_step", description="Execute a single step from the plan")
    def execute_step(self, step: Annotated[str, "Step to execute"]) -> str:
        return f"Executed: {step} - Completed successfully."

# Add plugins
kernel.add_plugin(PlanningPlugin(), plugin_name="planner")
kernel.add_plugin(SearchPlugin(), plugin_name="search")
kernel.add_plugin(ExecutionPlugin(), plugin_name="executor")

In [ ]:
async def planner_agent(goal: str) -> str:
    """Planning agent that breaks down and executes complex tasks."""
    
    chat_history = ChatHistory()
    chat_history.add_system_message(
        "You are a planning agent. Break down complex tasks into steps, "
        "search for needed information, and execute each step systematically. "
        "Use the available tools to create a plan, search for information, and execute steps."
    )
    chat_history.add_user_message(goal)
    
    execution_settings = AzureChatPromptExecutionSettings(
        service_id=service_id,
        function_choice_behavior=FunctionChoiceBehavior.Auto(),
    )
    
    chat_service = kernel.get_service(service_id)
    
    response = await chat_service.get_chat_message_content(
        chat_history=chat_history,
        settings=execution_settings,
        kernel=kernel,
    )
    
    return str(response)

In [ ]:
# Test planner agent
result = await planner_agent("Research and summarize the benefits of microservices architecture")
print("Planner Agent Result:")
print(result)

**Note:** The planner pattern in Semantic Kernel uses function calling to coordinate planning, information gathering (search), and execution steps. The LLM acts as an intelligent orchestrator that determines which tools to use and when.

For production use, integrate with real search APIs like Tavily or Bing Search for actual information retrieval.